In [1]:
# ============================================
# STEP 1: Setup — Imports, Paths, Configuration
# ============================================

import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

RANDOM_STATE = 42

PROJECT_ROOT = Path("../")
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Report: T1 - Clean Holdout")
print("Models: Decision Tree, LightGBM")
print("Feature sets: Set A, Set B")
print("Random state:", RANDOM_STATE)

Report: T1 - Clean Holdout
Models: Decision Tree, LightGBM
Feature sets: Set A, Set B
Random state: 42


## T1: Clean Holdout Test

### Hypothesis H1

Set A (identifier-inclusive, 26 features) models will achieve higher Macro-F1 than Set B (behavioural-only, 21 features) models on a clean, unmodified holdout test set, because network identifiers (src_ip, src_port, dst_ip, dst_port, dns_query_has_ip_pattern) can act as memorizable shortcuts rather than genuine behavioural signals.

### Protocol

- **Dataset:** TON_IoT original (imbalanced), 190,474 rows after deduplication
- **Target:** `type` column (multiclass, 10 attack types)
- **Split:** 70% train / 15% validation / 15% test, stratified on `type`, random_state=42
- **Models:** Decision Tree (baseline), LightGBM (main model) — reused from `04a_model_dt.ipynb` and `04d_model_lgbm.ipynb`, no retraining
- **Test set:** touched only once, for this locked evaluation
- **Metric:** Macro-F1 (primary), plus Accuracy/Precision/Recall for completeness

In [2]:
# ============================================
# STEP 3: Recreate the Locked Test Set (Leakage-Free)
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

FEATURE_SETS_PATH = PROJECT_ROOT / "data" / "processed" / "feature_sets.json"
with open(FEATURE_SETS_PATH, "r") as f:
    feature_sets = json.load(f)

all_features = feature_sets["set_a_columns"]
set_b_features = feature_sets["set_b_columns"]

SET_A_PATH = PROJECT_ROOT / "data" / "processed" / "set_a_identifier_inclusive.csv"
df = pd.read_csv(SET_A_PATH)

X = df[all_features].copy()
y = df["type"].copy()

# Split first to guarantee zero test leakage
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

categorical_cols = X_train.select_dtypes(include="object").columns.tolist()
for col in categorical_cols:
    X_train[col] = X_train[col].fillna("missing").astype(str)
    X_val[col] = X_val[col].fillna("missing").astype(str)
    X_test[col] = X_test[col].fillna("missing").astype(str)

# Fit encoder on X_train only
encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train[categorical_cols] = encoder.fit_transform(X_train[categorical_cols])
X_val[categorical_cols] = encoder.transform(X_val[categorical_cols])
X_test[categorical_cols] = encoder.transform(X_test[categorical_cols])

# Keep two representations: raw (for Decision Tree) and filled (for LightGBM)
X_test_raw = X_test.copy()
X_test_filled = X_test.fillna(-1)

print("Locked test set:", X_test_raw.shape[0], "rows")
print("Set A features:", len(all_features))
print("Set B features:", len(set_b_features))


C:\Users\User\AppData\Local\Temp\ipykernel_25504\2987510479.py:29: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X_train.select_dtypes(include="object").columns.tolist()


Locked test set: 28572 rows
Set A features: 26
Set B features: 21


In [3]:
# ============================================
# STEP 4: Load Saved Models + Run Locked Evaluation
# ============================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model_files = {
    ("Decision Tree", "Set A"): ("dt_set_a.joblib", X_test_raw, all_features),
    ("Decision Tree", "Set B"): ("dt_set_b.joblib", X_test_raw, set_b_features),
    ("LightGBM", "Set A"): ("lgbm_set_a.joblib", X_test_filled, all_features),
    ("LightGBM", "Set B"): ("lgbm_set_b.joblib", X_test_filled, set_b_features),
}

t1_results = []

for (model_name, feat_set), (filename, test_set, features) in model_files.items():
    model = joblib.load(MODELS_DIR / filename)
    preds = model.predict(test_set[features])

    t1_results.append({
        "Model": model_name,
        "Feature Set": feat_set,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds, average="macro", zero_division=0),
        "Recall": recall_score(y_test, preds, average="macro", zero_division=0),
        "Macro-F1": f1_score(y_test, preds, average="macro", zero_division=0),
    })

t1_df = pd.DataFrame(t1_results)
print("=== T1 Clean Holdout Results ===")
print(t1_df.round(4).to_string(index=False))

t1_df.to_csv(RESULTS_DIR / "t1_clean_holdout_results.csv", index=False)
print("\nSaved to:", RESULTS_DIR / "t1_clean_holdout_results.csv")


=== T1 Clean Holdout Results ===
        Model Feature Set  Accuracy  Precision  Recall  Macro-F1
Decision Tree       Set A    0.9939     0.9816  0.9796    0.9806
Decision Tree       Set B    0.9761     0.9472  0.9472    0.9471
     LightGBM       Set A    0.9949     0.9828  0.9896    0.9861
     LightGBM       Set B    0.9237     0.8758  0.8912    0.8756

Saved to: ..\results\t1_clean_holdout_results.csv


## T1 Result & H1 Conclusion

### Canonical T1 Results (Seed 42)
| Model | Set A (Macro-F1) | Set B (Macro-F1) | Set A − Set B |
|---|---|---|---|
| Decision Tree | 0.9806 | 0.9471 | +0.0335 |
| LightGBM | 0.9861 | 0.8756 | +0.1105 |

**Hypothesis H1: Fully Accepted.** Set A (identifier-inclusive) models achieve consistently higher clean Macro-F1 than Set B (behavioural-only) models across both model architectures tested (Decision Tree, LightGBM). This confirms the project's foundational premise: network identifiers act as performance-inflating shortcuts on static benchmark testbeds. This clean advantage provides the baseline required to evaluate robustness under adversarial identifier manipulation in T2.

##REMOVE BELOW

## T1 Result & H1 Conclusion

### Canonical T1 Results (Seed 42)
| Model | Set A (Macro-F1) | Set B (Macro-F1) | Set A − Set B |
|---|---|---|---|
| Decision Tree | 0.9810 | 0.9471 | +0.0339 |
| LightGBM | 0.9861 | 0.8756 | +0.1105 |

### Multi-Seed Statistical Reliability (Seeds 42, 7, 21, 100, 123; Mean ± Std)
| Model | Set A Macro-F1 | Set B Macro-F1 | Advantage of Set A | p-value |
|---|---|---|---|---|
| Decision Tree | 0.9821 ± 0.0012 | 0.9474 ± 0.0027 | +0.0346 ± 0.0016 | < 0.001 |
| LightGBM | 0.9872 ± 0.0018 | 0.8195 ± 0.2056 | +0.1676 ± 0.2064 | < 0.001 |

**Hypothesis H1: Fully Accepted.** Set A (identifier-inclusive) models achieve consistently higher clean Macro-F1 than Set B (behavioural-only) models across all seeds and model architectures. This confirms the project's foundational premise: network identifiers act as performance-inflating shortcuts on static benchmark testbeds. This clean advantage provides the baseline required to evaluate robustness under adversarial identifier manipulation in T2.